# Task 1.1 

In [6]:
import pandas as pd

data = pd.read_csv('winequality-red.csv', sep=';')

shuffled_data = data.sample(frac=1, random_state=42).reset_index(drop=True)

training_data = shuffled_data.sample(frac=0.8, random_state=42)
testing_data = shuffled_data.drop(training_data.index)


I decided to use a 80/20 split for training/testing, I will split my data in to 2 sets where one set contains 80% of the data and is used for training while the other set contains the remaining 20% and is used for testing the model trained on the other set.

In [7]:
import math

quality_scores = training_data['quality']

bad_quality = quality_scores[quality_scores <= 5].count()
good_quality = quality_scores[quality_scores > 5].count()
print("bad quality count:", bad_quality)
print("good quality count:", good_quality)

p_low = bad_quality / len(quality_scores)
p_high = good_quality / len(quality_scores)

root_entropy = - (p_low * math.log2(p_low) + p_high * math.log2(p_high))

print("Root entropy:", root_entropy)

bad quality count: 593
good quality count: 686
Root entropy: 0.9961827316335294


In [8]:
def calculate_split_points(feature):
    unique_values = feature.unique()
    unique_values.sort()
    split_points = []

    for i in range(len(unique_values) - 1):
        split_value = (unique_values[i] + unique_values[i + 1]) / 2
        split_points.append(float(split_value))
    return split_points

def calculate_entropy(partition):
    total_count = len(partition)
    if total_count == 0:
        return 0

    quality_scores = partition['quality']
    bad_quality = quality_scores[quality_scores <= 5].count()
    good_quality = quality_scores[quality_scores > 5].count()

    p_low = bad_quality / total_count
    p_high = good_quality / total_count

    if p_low == 0 or p_high == 0:
        return 0

    entropy = - (p_low * math.log2(p_low) + p_high * math.log2(p_high))
    return entropy

def calculate_information_gain(data, feature, split_point, parent_entropy):
    left_partition = data[data[feature] <= split_point]
    right_partition = data[data[feature] > split_point]

    total_count = len(data)
    left_count = len(left_partition)
    right_count = len(right_partition)

    if total_count == 0:
        return 0
    
    weight_left = left_count / total_count
    weight_right = right_count / total_count
    gain = parent_entropy - (weight_left * calculate_entropy(left_partition) + weight_right * calculate_entropy(right_partition))

    return gain

def find_best_split(data, parent_entropy):
    best_gain = -1
    best_feature = None
    best_split_point = None
    
    for feature in data.columns[:-1]: 
        split_points = calculate_split_points(data[feature])

        for split_point in split_points:
            gain = calculate_information_gain(data, feature, split_point, parent_entropy)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_split_point = split_point
    return best_feature, best_split_point

def get_label(data):
    quality_scores = data['quality']
    if (quality_scores > 5).sum() >= (quality_scores <= 5).sum():
        return "good"
    else:
        return "bad"
    
def train(data):

    entropy = calculate_entropy(data)

    if entropy == 0:
        return {"label": get_label(data)}

    feature, split = find_best_split(data, entropy)

    if feature is None:
        return {"label": get_label(data)}

    left_data = data[data[feature] <= split]
    right_data = data[data[feature] > split]

    if len(left_data) == 0 or len(right_data) == 0:
        return {"label": get_label(data)}

    left_tree = train(left_data)
    right_tree = train(right_data)

    return {
        "feature": feature,
        "split": split,
        "left": left_tree,
        "right": right_tree
    }
tree = train(training_data)

def print_tree(tree, depth=0):
    indent = "  " * depth

    if "label" in tree:
        print(indent + "→ " + tree["label"])
        return

    feature = tree["feature"]
    split = tree["split"]

    print(indent + f"[{feature} <= {split}]")

    print(indent + "L:", end="")
    print_tree(tree["left"], depth + 1)

    print(indent + "R:", end="")
    print_tree(tree["right"], depth + 1)

print_tree(tree)

[alcohol <= 10.25]
L:  [sulphates <= 0.575]
  L:    [total sulfur dioxide <= 114.5]
    L:      [chlorides <= 0.0795]
      L:        [pH <= 3.275]
        L:          [total sulfur dioxide <= 73.5]
          L:            [alcohol <= 10.05]
            L:              [residual sugar <= 1.55]
              L:                → good
              R:                [fixed acidity <= 8.3]
                L:                  → bad
                R:                  [fixed acidity <= 8.55]
                  L:                    → good
                  R:                    [chlorides <= 0.0535]
                    L:                      → good
                    R:                      [free sulfur dioxide <= 11.0]
                      L:                        → bad
                      R:                        [volatile acidity <= 0.445]
                        L:                          → bad
                        R:                          [volatile acidity <= 0.76]
        

The output of the trained decision tree is in dictionary format where each node in the tree represents an if-else rule with a split value based on the most information gain, each rule has nested left and right dictionaries containing partitions based on the split, the resulting tree very large, this is due to splitting until only leaves of one class remain, this is not ideal as it will recursively call the train function resulting in a very deep nested tree that is overtrained on one set of data. 

The label 'quality' was continuous so I decided to group the labels into 'bad quality' = quality < 5 and 'good quality' = quality >= 5. I chose to have 2 labels as it allows me to have binary classification and use a classfication based decision tree. 

# Task 1.2

In [9]:
def train(data, stopping_depth, current_depth=0):

    entropy = calculate_entropy(data)

    if entropy == 0 or len(data) <= 1 or current_depth >= stopping_depth:
        return {"label": get_label(data)}

    feature, split = find_best_split(data, entropy)

    if feature is None:
        return {"label": get_label(data)}

    left_data = data[data[feature] <= split]
    right_data = data[data[feature] > split]

    if len(left_data) == 0 or len(right_data) == 0:
        return {"label": get_label(data)}

    left_tree = train(left_data, stopping_depth, current_depth + 1)
    right_tree = train(right_data, stopping_depth, current_depth + 1)

    return {
        "feature": feature,
        "split": split,
        "left": left_tree,
        "right": right_tree
    }
tree_depth2 = train(training_data, stopping_depth=2)
tree_depth3 = train(training_data, stopping_depth=3)
tree_depth4 = train(training_data, stopping_depth=4)
print("Tree with stopping depth 2:")
print_tree(tree_depth2)
print()

print("Tree with stopping depth 3:")
print_tree(tree_depth3)
print()

print("Tree with stopping depth 4:")
print_tree(tree_depth4)


Tree with stopping depth 2:
[alcohol <= 10.25]
L:  [sulphates <= 0.575]
  L:    → bad
  R:    → bad
R:  [alcohol <= 11.45]
  L:    → good
  R:    → good

Tree with stopping depth 3:
[alcohol <= 10.25]
L:  [sulphates <= 0.575]
  L:    [total sulfur dioxide <= 114.5]
    L:      → bad
    R:      → bad
  R:    [total sulfur dioxide <= 82.5]
    L:      → bad
    R:      → bad
R:  [alcohol <= 11.45]
  L:    [sulphates <= 0.585]
    L:      → bad
    R:      → good
  R:    [fixed acidity <= 6.95]
    L:      → good
    R:      → good

Tree with stopping depth 4:
[alcohol <= 10.25]
L:  [sulphates <= 0.575]
  L:    [total sulfur dioxide <= 114.5]
    L:      [chlorides <= 0.0795]
      L:        → bad
      R:        → bad
    R:      → bad
  R:    [total sulfur dioxide <= 82.5]
    L:      [volatile acidity <= 0.405]
      L:        → good
      R:        → bad
    R:      [residual sugar <= 11.75]
      L:        → bad
      R:        → good
R:  [alcohol <= 11.45]
  L:    [sulphates <= 0.5

By implementing tree depth control we can choose how deep our model should be, I trained 3 models with stopping levels 2, 3, 4 respectively. From the results of the models we see that the root rules for each model are the same and for each level they remain the same, however as the depth increases new rules are found that further splits and classifies the data with greater depth.

When comparing the models with stopping depth 3 compared to 2 we see the left side of the tree has no change in classification and still predicts 'bad' however the right side 'alcohol <= 11.45' has one new rule 'sulphates <= 0.585' that changes the classfication from 'good' to 'bad' if 'sulphates <= 0.585' is false.

The model with stopping depth 4 reveals more rules and the classification change again. It shows we can classify data with more detail by increasing depth and adding rules however a higher depth can result in overfitting so we need to find a optimal depth that increases accuracy without overfitting.

# Task 1.3

In [10]:
def predict(tree, row):
    if "label" in tree:
        return tree["label"]

    feature = tree["feature"]
    split = tree["split"]

    if row[feature] <= split:
        return predict(tree["left"], row)
    else:
        return predict(tree["right"], row)

def accuracy(tree, testing_data):
    correct_predictions = 0
    for index, row in testing_data.iterrows():
        predicted_label = predict(tree, row)
        actual_label = "good" if row['quality'] > 5 else "bad"
        if predicted_label == actual_label:
            correct_predictions += 1
    return correct_predictions / len(testing_data)

tree_depth4_accuracy = accuracy(tree_depth4, testing_data)
tree_accuracy = accuracy(tree, testing_data)
print(f"Accuracy of the tree with stopping depth 4 on the testing set: {tree_depth4_accuracy:.2%}")
print(f"Accuracy of the tree with no stopping depth on the testing set: {tree_accuracy:.2%}")

Accuracy of the tree with stopping depth 4 on the testing set: 72.19%
Accuracy of the tree with no stopping depth on the testing set: 76.25%


The tree with no stopping depth achieved higher accuracy on the test set (76.25%) compared to the tree with stopping depth 4 (72.19%). This could be due to the limited test data or the classification approach - I chose to classify the label 'quality' into groups which could make the data simpler and easier for the no stopping depth tree to memorise and avoid overfitting. Although the no stopping depth tree has greater accuracy it may not perform well on new useen data due to potential overfitting, the depth controlled model is generalized providing a simpler model even though accuracy is slightly lower. HOwever it could have some risk of underfitting so the best model would have some tradeoff between underfitting and overfitting.

# 2.1

The splitting criterion determines the best feature and split point of each node in the decision tree. Information gain is the current criterion, it is calculated by finding the maximum reduction in entropy between parent and child nodes, if we change the splitting criterion to another criteron such as classification error it could affect the rules of the tree which leads to potential changes in structure and depth of the tree, the accuracy of the new tree could also be different due to the modified tree structure.

Classification error splits based on the majority class while information gain is based on entropy, it doesn't care about the impurity of data but rather reducing misclassfication of the majority class. This could result in a more simple shallow tree in some cases as it ignores the minority class and stops splitting if the majority class dominates. It may also cause the tree to be less accurate than information gain in some datasets.

# 2.2

Explain whether your test procedure implemented in Task 1-C can indicate whether your tree is over- or underfitting.  

By training multiple models with increasing depth and using the accuracy classifier evaluation metric to test the models we can monitor the increase in accuracy as the depth increases and determine the point at which accuracy does not increase by a meaningful amount. If we keep increasing the depth but there is no noticable increase in accuracy, It can indicate that the model is potentially overfitted, the opposite can also indicate the model may be underfitted if the accuracy steadily increases along with depth. By monitoring the test results across multiple models with incrementive depth we can determine a appropriate fit for our model.